In [1]:
from src.basemodel import Capa, Clasificacion,Segmento,Articulo,Familia,Clase
import os
import pandas as pd
from dotenv import load_dotenv
from google import genai
import psycopg2
from src.database import create_connection
from src.ai import generate_prompt,generar_familias_prompt
import json

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_IA_MODEL = os.getenv("GEMINI_IA_MODEL")

client = genai.Client(api_key=GEMINI_API_KEY)

segmentos:list[Segmento] = []
articulos: list[Articulo] = []
familias: list[Familia] = []
clases:list[Clase] = []

capa1_segmentos:dict[str,Capa[Segmento]] = {}
capa2_familias:dict[str,Capa[Familia]] = {}
capa3_clases:dict[str,Capa[Clase]] = {}
capa4_articulos:dict[str,list[Articulo]] = {}

catalogo: dict[str,list[Segmento]] = []

articulos = [
    Articulo("adajk","coca cola 1/2"),
    Articulo("calmkllwa","gaseosa kr 300 ml"),
    Articulo("aadac","cuarto de pollo"),
    Articulo("alwekmav","delivery domicilio"),
    Articulo("acmwakoi","chaufa + pollo"),
    Articulo("aaasdc","tallarin saltado")
]


In [2]:
conn = create_connection()
with conn.cursor() as cur:
    cur.execute("select id,descripcion from segmento")
    result = cur.fetchall()
conn.close()
segmentos = [Segmento(id,nombre) for id,nombre in result]

In [3]:
prompt = generate_prompt(
        (seg.nombre for seg in segmentos),
        (art.nombre for art in articulos)
    )
print(prompt)

response = client.models.generate_content(
    contents=prompt,
    model=GEMINI_IA_MODEL,
    config={
        "response_mime_type":"application/json",
        "response_schema":list[Clasificacion]
    }
)
data:dict = json.loads(response.text)


Eres un clasificador del catálogo SUNAT.

Debes clasificar cada artículo en UNO de los segmentos.

Segmentos:

0: Material Vivo Vegetal y Animal, Accesorios y Suministros
1: Material Mineral, Textil y  Vegetal y Animal No Comestible
2: Material Químico incluyendo Bioquímicos y Materiales de Gas
3: Materiales de Resina, Colofonia, Caucho, Espuma, Película y Elastómericos
4: Materiales y Productos de Papel
5: Materiales Combustibles, Aditivos para Combustibles, Lubricantes y Anticorrosivos
6: Maquinaria y Accesorios de Minería y Perforación de Pozos
7: Maquinaria y Accesorios para Agricultura, Pesca, Silvicultura y Fauna
8: Maquinaria y Accesorios para Construcción y Edificación
9: Maquinaria y Accesorios para Manufactura y Procesamiento Industrial
10: Maquinaria, Accesorios y Suministros para Manejo, Acondicionamiento y Almacenamiento de Materiales
11: Vehículos Comerciales, Militares y Particulares, Accesorios y Componentes
12: Maquinaria y Accesorios para Generación y Distribución de

In [4]:
print(response.text)
dict_articulos = {}
for i,articulo in enumerate(articulos):
    dict_articulos[i] = articulo
print(dict_articulos)

[
  {
    "id_articulo": 0,
    "id_grupo": 28,
    "confianza": 0.99
  },
  {
    "id_articulo": 1,
    "id_grupo": 28,
    "confianza": 0.99
  },
  {
    "id_articulo": 2,
    "id_grupo": 28,
    "confianza": 0.99
  },
  {
    "id_articulo": 3,
    "id_grupo": 42,
    "confianza": 0.95
  },
  {
    "id_articulo": 4,
    "id_grupo": 28,
    "confianza": 0.99
  },
  {
    "id_articulo": 5,
    "id_grupo": 28,
    "confianza": 0.99
  }
]
{0: Articulo(adajk), 1: Articulo(calmkllwa), 2: Articulo(aadac), 3: Articulo(alwekmav), 4: Articulo(acmwakoi), 5: Articulo(aaasdc)}


In [5]:

#for capa in capa1_segmentos.values():capa.items = []

In [6]:
capa1_segmentos = {}
for match in data:
    indexSegmento = int(match.get("id_grupo"))
    segmento = segmentos[indexSegmento]
    articulo = articulos[int(match.get("id_articulo"))]
    capa1_segmentos.setdefault(segmento.id,Capa(segmento,indexSegmento))
    print(segmento,articulo)
    capa1_segmentos.get(segmento.id).items.append(articulo)

for capa in capa1_segmentos.values():print(len(capa.items))

Segmento(50000000) Articulo(adajk)
Segmento(50000000) Articulo(calmkllwa)
Segmento(50000000) Articulo(aadac)
Segmento(78000000) Articulo(alwekmav)
Segmento(50000000) Articulo(acmwakoi)
Segmento(50000000) Articulo(aaasdc)
6
6


In [10]:
segmento_ids = tuple(id[:2] for id in capa1_segmentos.keys())
query = """ select id,descripcion from familia where {} """.format(" OR ".join(["id like %s"] * len(segmento_ids)))
params = tuple(f"{id}%" for id in segmento_ids)
conn = create_connection()
with conn.cursor() as cur:
    cur.execute(query,params)
    result = cur.fetchall()
conn.close()
familias = [Familia(id,nombre) for id,nombre in result]

In [11]:
for capa in capa1_segmentos.values(): capa.value.familias = []
for fam in familias:
    tramoSegmento = fam.id[:2]
    segmento = next((s.value for s in capa1_segmentos.values() if s.value.id.startswith(tramoSegmento)),None)
    if segmento is not None: segmento.familias.append(fam)

In [20]:
data: dict = {}
for seg in capa1_segmentos.values():
    prompt = generar_familias_prompt(
        [fam.nombre for fam in seg.value.familias],
        [art.nombre for art in seg.items],
    )
    print(prompt)
    response = client.models.generate_content(
        contents=prompt,
        model=GEMINI_IA_MODEL,
        config={
            "response_mime_type":"application/json",
            "response_schema":list[Clasificacion]
        }
    )
    data[seg.value.id] = response


Eres un clasificador del catálogo SUNAT.

Debes clasificar cada artículo en UNA de las familias.

familias:

0: Frutos secos
1: Productos de carne y aves de corral
2: Pescados y mariscos
3: Productos lácteos y huevos
4: Aceites y grasas comestibles
5: Chocolates, azúcares, edulcorantes y productos de confitería
6: Condimentos y conservantes
7: Productos de panadería
8: Alimentos preparados y conservados
9: Bebidas
10: Tabaco y productos de fumar y substitutos
11: Productos de cereales y legumbres
12: Fruta fresca
13: Fruta orgánica fresca
14: Fruta seca
15: Fruta orgánica seca
16: Fruta congelada
17: Fruta orgánica congelada
18: Fruta en lata o en frasco
19: Fruta orgánica en lata o en frasco
20: Puré de frutas
21: Vegetales frescos
22: Vegetales orgánicos frescos
23: Vegetales secos
24: Vegetales orgánicos secos
25: Vegetales congelados
26: Vegetales orgánicos congelados
27: Vegetales en lata o en frasco
28: Vegetales orgánicos en lata o en frasco

Artículos:

id_articulo: 0, nombre:

In [18]:
print([res.text for res in data.values()])

['[{"id_articulo": 0, "id_grupo": 9, "confianza": 0.99}, {"id_articulo": 1, "id_grupo": 9, "confianza": 0.99}, {"id_articulo": 2, "id_grupo": 1, "confianza": 0.95}, {"id_articulo": 3, "id_grupo": 8, "confianza": 0.85}, {"id_articulo": 4, "id_grupo": 8, "confianza": 0.95}, {"id_articulo": 5, "id_grupo": 8, "confianza": 0.95}]', '[\n  {\n    "id_articulo": 0,\n    "id_grupo": 2,\n    "confianza": 0.85\n  },\n  {\n    "id_articulo": 1,\n    "id_grupo": 2,\n    "confianza": 0.85\n  },\n  {\n    "id_articulo": 2,\n    "id_grupo": 2,\n    "confianza": 0.85\n  },\n  {\n    "id_articulo": 3,\n    "id_grupo": 0,\n    "confianza": 0.95\n  },\n  {\n    "id_articulo": 4,\n    "id_grupo": 2,\n    "confianza": 0.85\n  },\n  {\n    "id_articulo": 5,\n    "id_grupo": 2,\n    "confianza": 0.85\n  }\n]']
